In [ ]:
import sys
sys.path.append('/home/salzmann/Desktop/einc/cluster_home/ultracold-dipolar/Own_program_vault')
import main_version_2 as mv2
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
from matplotlib import colors
import os
import gc

fontsize = 15
plt.rcParams["image.origin"] = 'lower'
plt.rcParams['legend.handlelength'] = 0.5


pgf_with_rc_fonts = {
    "font.family": "serif",
    "font.serif": [],
    "font.sans-serif": ["DejaVu Sans"]
}
plt.rcParams.update(pgf_with_rc_fonts)

plt.rc('font', **{'family': 'serif', 'serif': ['Computer Modern Roman'],
                  'size': fontsize})
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{amsmath} \usepackage{amsfonts}')
plt.rc('legend', fontsize=fontsize,
       title_fontsize=fontsize)


# Get the current working directory (where the notebook is running)
current_directory = os.getcwd()


In [ ]:
import re
ground_state_files = []
ground_state_paths = []
name_pattern = re.compile(r"ground_state_wave_function.vtk$")

for root, dirs, files in os.walk(current_directory, topdown=True):
    for filename in files:
        if name_pattern.match(filename):
            print(filename)

In [ ]:
import re
ground_state_files = []
ground_state_paths = []
name_pattern = re.compile(r"ground_state_wave_function.vtk$")

for root, dirs, files in os.walk(current_directory, topdown=True):
    for filename in files:
        if name_pattern.match(filename):
            full_path = os.path.join(root, filename)
            ground_state_files.append(full_path)
            ground_state_paths.append(root)

ground_state_paths_sorted = sorted(ground_state_paths, key=lambda x: x[-29:])

ground_state_files_sorted = sorted(ground_state_files, key=lambda p: float(re.search(r"edd_([0-9.]+)", p).group(1)))


In [ ]:
r_measure = 10.185
y_coord = int(384/2 - r_measure/24*384)
x_coord = 384-y_coord

In [ ]:
superfluid_ratios = []
edge_fraction = []
summed_up_number_of_particles = []
integrated_number_density = []
for i in range(len(ground_state_files_sorted)):
    print(i, ground_state_files_sorted[i])
    proc = mv2.TestProcessing.Three_D(file=ground_state_files_sorted[i], iteration=0)
    proc.print_out_stuff()
    realspace_density = proc.density_real_space()
    grid_spacing = proc.grid_spacing
    momentumspace_density = proc.density_momentum_space()
    keep = np.s_[:,:,192]
    proc.PLOT_Two_D_slice(x_dot=x_coord, y_dot=y_coord, r_input=r_measure, save=True, index_tuple = keep)
    mask = proc.cylindrical_mask(center = None ,radius_1= 120, radius_2= 160) 
    mask = mask.astype(bool)  
    keep = np.s_[:,:,192]
    proc.PLOT_Two_D_slice(x_dot=x_coord, y_dot=y_coord, r_input=r_measure, save=False, index_tuple = keep, radial_mask = mask)

    masked_density_full = np.where(mask[:, :, None], realspace_density, 0)
    masked_manifold = np.where(masked_density_full<10, 0.0,1.0)
    alpha = masked_manifold[:,:,192]
    masked_volume=np.sum(masked_manifold)*grid_spacing[0]*grid_spacing[1]*grid_spacing[2]
    masked_number_of_particles=np.sum(masked_density_full)*grid_spacing[0]*grid_spacing[1]*grid_spacing[2]
    masked_number_density=masked_number_of_particles/masked_volume

    asdfa, full_vol_container = proc.Leggetts_integral(fxyz=masked_manifold, nr=512, ntheta=512, delta_z = grid_spacing[2], center=None, with_radial_profile=False)
    Legget_integral, integrated_number, radial_profile = proc.Leggetts_integral(fxyz=masked_density_full, nr=512, ntheta=512, delta_z = grid_spacing[2], center=None, with_radial_profile=True)

    integrated_density = integrated_number/full_vol_container


    Leggets_estimate = 1/(integrated_number*Legget_integral)

    print("integrated density: ", integrated_density)
    print("summed up density in the volume: ", masked_number_density)
    print("superfluid ratio according to Leggets estimate",Leggets_estimate*2*np.pi*2*np.pi)

    superfluid_ratios.append(Leggets_estimate*2*np.pi*2*np.pi)
    integrated_number_density.append(integrated_density)
    summed_up_number_of_particles.append(masked_number_of_particles)
    edge_fraction.append(masked_number_of_particles/proc.number_of_particles)
    plt.plot( np.linspace(0, 2*np.pi, radial_profile.shape[0]), radial_profile)
    plt.xlabel('Angle [rad]')
    plt.ylabel('Density')
    plt.title('Radial profile of the density')


In [ ]:
edd_values = [float(re.search(r"edd_([0-9.]+)", path).group(1)) for path in ground_state_files_sorted]


In [ ]:
edd_values

In [ ]:
edge_fraction

In [ ]:
import csv

# Pair edd values with corresponding superfluid_ratios
results = list(zip(edd_values, superfluid_ratios, edge_fraction))

# Save the results to a CSV file
with open("data_N_100k_mit_cutoff.csv", "w", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["edd_value", "superfluid_ratio"])
    writer.writerows(results)